# Field transformations

`FieldTransformation(source, expr)` is the analogue of a FeynRules `Definitions`
entry. Transformations act on **compiled** interaction terms: covariant
derivatives and field strengths are expanded in the original gauge basis first.

The Standard Model workflow is:

1. compile the gauge-basis Lagrangian,
2. expand finite weak indices,
3. apply one simultaneous transformation stage,
4. extract physical-basis Feynman rules.


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_FUND_INDEX,
    CompiledLagrangian,
    DC,
    Field,
    FieldTransformation,
    GaugeGroup,
    Model,
    Parameter,
    PartialD,
    ProjM,
    find_source_basis_occurrences,
    flavor_index,
    replacement,
    rotation,
    validate_compiled_index_multiplicities,
)
from feynpy.interactions import InteractionTerm
from models.SM import build_standard_model
from models.SM.SM_support import standard_model_weak_tensor_components


## Linear mixing after compilation

A Higgs kinetic term is compiled with the hypercharge field `B`. The
transformation `B → -s_w Z + c_w A` is applied afterwards.


In [2]:
mu, g1, sw, cw, yH = S("mu"), S("g1"), S("sw"), S("cw"), S("yH")

B = Field("B", spin=1, self_conjugate=True, symbol=S("B0"), indices=(LORENTZ_INDEX,))
A = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Z = Field("Z", spin=1, self_conjugate=True, symbol=S("Z0"), indices=(LORENTZ_INDEX,))
Hdoublet = Field(
    "Hdoublet",
    spin=0,
    self_conjugate=False,
    symbol=S("Hdoublet0"),
    conjugate_symbol=S("Hdoubletdag0"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": yH},
)
U1Y = GaugeGroup(name="U1Y", abelian=True, coupling=g1, gauge_boson="B", charge="Y")
source_model = Model(
    DC(Hdoublet.bar, mu) * DC(Hdoublet, mu),
    gauge_groups=(U1Y,),
    fields=(Hdoublet, B, A, Z),
)
show_model(source_model, Hdoublet.bar, Hdoublet, B)

L_physical = source_model.transform_fields(FieldTransformation(B, -sw * Z + cw * A), repeat=False)
for vector in (B, A, Z):
    try:
        rule = L_physical.feynman_rule(Hdoublet.bar, Hdoublet, vector, include_delta=False)
    except ValueError:
        rule = 0
    show(f"Γ(H†, H, {vector.name}) after mixing", rule)


Lagrangian
DC(Hdoublet.bar, mu) * DC(Hdoublet, mu)

Feynman Rule
-1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Γ(H†, H, B) after mixing
0

Γ(H†, H, A) after mixing
-1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g1*cw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Γ(H†, H, Z) after mixing
1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu3)-1𝑖*g1*sw*yH*g(cof(2, w1),cof(2, w2))*pcomp(q2,mu3)



## Component rules and vacuum shifts

`components={0: 2}` matches a numeric value in index slot 0. Expand finite
indices with `expand_index_components(...)` first. The two rules below are
the SM Higgs-doublet definitions.


In [3]:
HALF = Expression.num(1) / Expression.num(2)
INV_SQRT2 = HALF**HALF
vev = S("vev")

Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("Phi0"),
    conjugate_symbol=S("Phidag0"),
    indices=(WEAK_FUND_INDEX,),
)
GP = Field("GP", spin=0, self_conjugate=False, symbol=S("GP0"), conjugate_symbol=S("GM0"))
Higgs = Field("Higgs", spin=0, self_conjugate=True, symbol=S("Higgs0"))
G0 = Field("G0", spin=0, self_conjugate=True, symbol=S("G00"))

scalar_source = Model(Phi.bar * Phi, fields=(Phi, GP, Higgs, G0)).lagrangian()
weak_expanded = scalar_source.expand_index_components(WEAK_FUND_INDEX)
show("terms after weak expansion", len(weak_expanded.terms))

broken = weak_expanded.transform_fields(
    FieldTransformation(Phi, -Expression.I * GP, components={0: 1}),
    FieldTransformation(
        Phi,
        vev * INV_SQRT2 + INV_SQRT2 * Higgs + Expression.I * INV_SQRT2 * G0,
        components={0: 2},
    ),
    repeat=False,
    real_symbols=(vev,),
)
show("Γ(GP.bar, GP)", broken.feynman_rule(GP.bar, GP, include_delta=False))
show("Γ(Higgs, Higgs)", broken.feynman_rule(Higgs, Higgs, include_delta=False))
show("Γ(G0, G0)", broken.feynman_rule(G0, G0, include_delta=False))


terms after weak expansion
2

Γ(GP.bar, GP)
1𝑖

Γ(Higgs, Higgs)
1𝑖

Γ(G0, G0)
1𝑖



## Projectors, rotations, and replacement products

`ProjM` acts on the spinor index; `rotation(V, Vdag)` acts on the flavor
index. Spectator indices are inherited. For products of replacement fields,
derivatives obey the Leibniz rule.


In [4]:
generation = flavor_index("GenerationDemo", 3, prefix="gd")
QL = Field(
    "QL",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("QL0"),
    conjugate_symbol=S("QLbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX, generation, COLOR_FUND_INDEX),
)
dq = Field(
    "dq",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("dq0"),
    conjugate_symbol=S("dqbar0"),
    indices=(SPINOR_INDEX, generation, COLOR_FUND_INDEX),
)
V = Parameter("Vdemo", indices=(generation, generation), complex_param=True, unitary_partner="Vdagdemo")
Vdag = Parameter("Vdagdemo", indices=(generation, generation), complex_param=True, unitary_partner="Vdemo")

matrix_source = CompiledLagrangian(
    terms=(
        InteractionTerm(coupling=1, fields=(QL(S("s"), 2, S("f"), S("c")),)),
        InteractionTerm(coupling=1, fields=(QL.bar(S("sb"), 2, S("fb"), S("cb")),)),
    ),
    parameters=(V, Vdag),
)
matrix_result = matrix_source.transform_fields(
    FieldTransformation(QL, rotation(V, Vdag) * ProjM * dq, components={1: 2}),
    repeat=False,
)
show("matrix transformation couplings", [clean(term.coupling) for term in matrix_result.terms])

phi = Field("phi", spin=0, self_conjugate=True, symbol=S("phi0"))
x = Field("x", spin=0, self_conjugate=True, symbol=S("x0"))
y = Field("y", spin=0, self_conjugate=True, symbol=S("y0"))
leibniz = (
    Model(PartialD(phi, mu), fields=(phi, x, y))
    .lagrangian()
    .transform_fields(FieldTransformation(phi, terms=(replacement(1, x, y),)), repeat=False)
)
show("Leibniz on PartialD(phi) → x y", leibniz.to_symbolica())


matrix transformation couplings
['PL(s,i_canon_1)*Vdemo(f,gd_canon_1)', 'PR(i_canon_1,sb)*Vdagdemo(gd_canon_1,fb)']

Leibniz on PartialD(phi) → x y
x0*PartialD(y0,mu)+y0*PartialD(x0,mu)



## Simultaneous passes

`repeat=False` is one FeynRules-style definitions block. `repeat=True` walks
dependent stages to a fixed point.


In [5]:
stage_a = Field("stage_a", spin=0, self_conjugate=True, symbol=S("stage_a0"))
stage_b = Field("stage_b", spin=0, self_conjugate=True, symbol=S("stage_b0"))
stage_c = Field("stage_c", spin=0, self_conjugate=True, symbol=S("stage_c0"))
stage_source = Model(stage_a, fields=(stage_a, stage_b, stage_c)).lagrangian()
rules = (FieldTransformation(stage_a, stage_b), FieldTransformation(stage_b, stage_c))
show("repeat=False", stage_source.transform_fields(*rules, repeat=False).to_symbolica())
show("repeat=True", stage_source.transform_fields(*rules, repeat=True).to_symbolica())


repeat=False
stage_b0

repeat=True
stage_c0



## Standard Model usage

`models/SM` applies this pipeline to each sector: expand weak indices, then
one simultaneous `transform_fields(..., repeat=False)` pass.


In [6]:
sm = build_standard_model()
show(
    "gauge-basis → physical-basis definitions",
    "\n".join(
        f"{transformation.source.name}: {clean(transformation.expr)}"
        for transformation in sm.transformations
    ),
)

residual = find_source_basis_occurrences(
    sm.lagrangian,
    source_fields=(
        sm.fields.LL, sm.fields.lR, sm.fields.QL, sm.fields.uR, sm.fields.dR,
        sm.fields.Phi, sm.fields.B, sm.fields.Wi, sm.fields.ghB, sm.fields.ghWi,
    ),
)
show("compiled physical terms", len(sm.lagrangian.terms))
show("residual source fields", len(residual))
show("index-multiplicity issues", len(validate_compiled_index_multiplicities(sm.lagrangian)))


gauge-basis → physical-basis definitions
B: -sw * Z + cw * A
Wi: (1/2)^(1/2) * W.bar + (1/2)^(1/2) * W
Wi: -1𝑖*(1/2)^(1/2) * W.bar + 1𝑖*(1/2)^(1/2) * W
Wi: cw * Z + sw * A
Phi: -1𝑖 * GP
Phi: (1/2)^(1/2)*vev + (1/2)^(1/2) * H + 1𝑖*(1/2)^(1/2) * G0
LL: ProjM * vl
LL: ProjM * l
lR: ProjP * l
QL: ProjM * uq
QL: CKM * ProjM * dq
uR: ProjP * uq
dR: ProjP * dq
ghB: -sw * ghZ + cw * ghA
ghWi: (1/2)^(1/2) * ghWp + (1/2)^(1/2) * ghWm
ghWi: 1𝑖*(1/2)^(1/2) * ghWp + -1𝑖*(1/2)^(1/2) * ghWm
ghWi: cw * ghZ + sw * ghA

compiled physical terms
306

residual source fields
0

index-multiplicity issues
0

